# Evals — Main Agent

An eval is **test data + a grader + a loop**. Nothing more.

The test data is the course file `sms_conversations.json`. Every **recruiter**
turn carries a human label — `continue`, `schedule` or `end`. Candidate turns
carry `null`: they are input, not targets.

One eval item = the conversation history up to a labelled recruiter turn, and
the label that turn should get. The model never sees the turn it is predicting.

Order of work, and it matters: **the baseline is established before the model is
touched.** Without a number written down beforehand there is nothing to prove
afterwards.


In [ ]:
import json, sys, warnings
from collections import Counter, defaultdict
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from app.modules import config

conversations = json.loads(config.CONVERSATIONS_JSON.read_text(encoding="utf-8"))
print(f"{len(conversations)} conversations")
print(f"{sum(len(c['turns']) for c in conversations)} turns in total")

## 1. Build the test set

In [ ]:
def build_test_set(conversations):
    """One item per labelled recruiter turn: the history before it, and its label."""
    items = []
    for conversation in conversations:
        turns = conversation["turns"]
        for index, turn in enumerate(turns):
            if turn["label"]:
                items.append({
                    "conversation_id": conversation["conversation_id"],
                    "turn_id": turn["turn_id"],
                    "start_time_utc": conversation["start_time_utc"],
                    "history": turns[:index],
                    "correct_label": turn["label"],
                })
    return items

test_data = build_test_set(conversations)
distribution = Counter(item["correct_label"] for item in test_data)

print(f"eval items: {len(test_data)}")
for label, count in distribution.most_common():
    print(f"  {label:<9} {count:>3}  ({count/len(test_data):.1%})")

## 2. Baseline — before any model runs

The question a baseline answers: *how well does the dumbest possible system do?*
Here that is answering `continue` every single time.

In [ ]:
majority_label, majority_count = distribution.most_common(1)[0]
baseline = majority_count / len(test_data)

print(f"baseline  (always '{majority_label}'):  {baseline:.2%}")
print(f"random    (1 of 3):                  {1/3:.2%}")
print()
print("The model has to beat the first number to be worth anything.")

## 3. Ceiling — the labels disagree with themselves

Before blaming the model for errors, check whether the test data is answerable.
Identical candidate messages that lead to different labels put a hard cap on
accuracy: whichever way the model answers, one of them is marked wrong.

In [ ]:
groups = defaultdict(list)
for item in test_data:
    last_candidate = next((turn["text"] for turn in reversed(item["history"])
                           if turn["speaker"] == "candidate"), None)
    if last_candidate:
        groups[last_candidate.strip().lower()].append(item)

unavoidable_errors = 0
for text, items in groups.items():
    labels = Counter(i["correct_label"] for i in items)
    if len(labels) > 1:
        unavoidable_errors += len(items) - max(labels.values())
        where = ", ".join(f"conv {i['conversation_id']}/turn {i['turn_id']}"
                          f" = {i['correct_label']}" for i in items)
        print(f'"{text[:58]}..."')
        print(f"    {where}\n")

ceiling = (len(test_data) - unavoidable_errors) / len(test_data)
print(f"unavoidable errors: {unavoidable_errors}")
print(f"ceiling: {ceiling:.2%} — a perfect model cannot score higher")

## 4. The grader

`normalize` pulls one of the three labels out of whatever the model actually
said, so `"schedule."` and `"The answer is schedule"` both count. `grade` is a
string check, operation `eq`.

In [ ]:
from app.modules.main_agent import normalize

def grade(prediction, correct_label):
    """string_check - operation: eq"""
    return prediction == correct_label

for sample in ["schedule", "Schedule.", "The recruiter should continue.", "END", "maybe"]:
    print(f"{sample!r:<35} -> {normalize(sample)!r}")

## 5. A reference without a model

Before writing a prompt, it is worth knowing how far plain rules get on this
data. `app/modules/rule_reference.py` is seven regular expressions. It costs
nothing and runs instantly.

In [ ]:
from app.modules import rule_reference

rule_pred = [rule_reference.predict(item["history"]) for item in test_data]
rule_true = [item["correct_label"] for item in test_data]
rule_accuracy = sum(p == t for p, t in zip(rule_pred, rule_true)) / len(rule_true)

print(f"baseline (always continue)  {baseline:.2%}")
print(f"rules, no model             {rule_accuracy:.2%}")
print(f"ceiling                     {ceiling:.2%}")
print()
print("This is the number the prompt has to beat. A model scoring below it is")
print("not facing a hard task — it is being asked the wrong question.")

## 6. The eval loop

Three prompt versions, scored against each other in one run.

- **v1** — the obvious reading of the three labels
- **v2** — v1 plus the decision rules the data actually follows
- **v3** — v2 plus paraphrased few-shot examples

The examples in v3 are written by hand, never copied from this test set, so
the comparison stays honest.

Cost: 3 × 59 requests, a few minutes and a few cents. `temperature=0` makes the
run repeatable.

In [ ]:
from app.modules import main_agent

llm = main_agent.build_llm()
print("model:", config.CHAT_MODEL)

results = {}

for version, prompt in config.MAIN_AGENT_PROMPTS.items():
    predictions = []
    for index, item in enumerate(test_data):
        messages = main_agent.turns_to_messages(item["history"], prompt)
        if len(messages) == 1:
            prediction = "continue"            # nothing has been said yet
        else:
            prediction = normalize(llm.invoke(messages).content)
        predictions.append(prediction)
        if index % 20 == 0:
            print(f"  {version}  {index} / {len(test_data)}")
    results[version] = predictions
    hits = sum(p == t for p, t in zip(predictions, rule_true))
    print(f"  {version}  ->  {hits/len(rule_true):.2%}\n")

print("done")

In [ ]:
summary = pd.DataFrame([
    {"approach": "baseline — always 'continue'", "accuracy": baseline},
    {"approach": "rules, no model",              "accuracy": rule_accuracy},
    *[{"approach": f"prompt {v}", "accuracy":
       sum(p == t for p, t in zip(results[v], rule_true)) / len(rule_true)}
      for v in results],
    {"approach": "ceiling (contradictory labels)", "accuracy": ceiling},
]).set_index("approach")

summary.style.format({"accuracy": "{:.2%}"})

In [ ]:
# everything below uses the best prompt
best = max(results, key=lambda v: sum(p == t for p, t
                                      in zip(results[v], rule_true)))
y_true, y_pred = rule_true, results[best]
print(f"best prompt: {best}  ->  put MAIN_AGENT_PROMPT_VERSION={best} in .env")

## 7. Metrics for the winning prompt

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_true, y_pred)
passed = sum(grade(p, t) for p, t in zip(y_pred, y_true))

print(f"passed   {passed} / {len(y_true)}")
print(f"accuracy {accuracy:.2%}")
print(f"baseline {baseline:.2%}   ->  {accuracy - baseline:+.2%}")
print(f"ceiling  {ceiling:.2%}")
print()
print(classification_report(y_true, y_pred, labels=config.LABELS, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

labels = sorted(set(y_true) | set(y_pred))      # union: the model may invent a label
cm = confusion_matrix(y_true, y_pred, labels=labels)
df_cm = pd.DataFrame(cm, index=labels, columns=labels)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("predicted")
plt.ylabel("true")
plt.title(f"Main Agent — accuracy {accuracy:.1%}")
plt.tight_layout()
plt.show()

df_cm

Read the matrix **by rows**: a row is one true label, spread across what the
model predicted for it.

In [ ]:
rows = []
for index, label in enumerate(labels):
    TP = cm[index, index]
    FP = cm[:, index].sum() - TP
    FN = cm[index, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)
    rows.append({"label": label, "TP": TP, "FP": FP, "FN": FN, "TN": TN,
                 "recall": TP / (TP + FN) if TP + FN else 0,
                 "precision": TP / (TP + FP) if TP + FP else 0})

pd.DataFrame(rows).set_index("label").round(3)

## 8. Look at the failures

The number is not the output of an eval. The failures are — they are what the
next prompt change is based on.

In [ ]:
errors = [{"conv": item["conversation_id"],
           "turn": item["turn_id"],
           "true": true,
           "pred": pred,
           "last_candidate_msg": next((t["text"] for t in reversed(item["history"])
                                       if t["speaker"] == "candidate"), "")[:70]}
          for item, true, pred in zip(test_data, y_true, y_pred) if true != pred]

print(f"{len(errors)} failures")
pd.DataFrame(errors)

## 9. Exit Advisor — before and after fine-tuning

The Main Agent decides *that* the conversation ends. The Exit Advisor decides
*how*, and the two endings in the data are opposites:

- 11 conversations end with a **confirmed interview**
- 4 end with the candidate **withdrawing** (conversations 5, 8, 13, 15)

A prompted base model tends to congratulate both. The cell below runs the base
model and the fine-tuned one side by side in a single pass, so nothing has to
be remembered between runs.

Fine-tuning route: **local, Hugging Face**. OpenAI's platform is closed to new
accounts, so `scripts/finetune_exit_advisor.py` trains a LoRA adapter on
Qwen2.5-0.5B-Instruct and `LOCAL_EXIT_MODEL` in `.env` points at the result.
Until that is set, the "tuned" column says so instead.

The validation file is hand-written and held out — the 4 refusals in the
course data are all in the training file, so scoring on those would be measuring
memorisation, not learning.

In [ ]:
from app.modules import exit_advisor

validation = [json.loads(line) for line in
              (config.DATA_DIR / "exit_advisor_valid.jsonl").read_text(
                  encoding="utf-8").splitlines() if line.strip()]

print("tuned backend:", exit_advisor.backend_name())
print(f"base  backend: openai:{config.CHAT_MODEL}\n")

comparison = []
for example in validation:
    history = [{"speaker": "recruiter" if m["role"] == "assistant" else "candidate",
                "text": m["content"]}
               for m in example["messages"][1:-1]]

    base = exit_advisor.close(history, force_base=True)
    tuned = (exit_advisor.close(history)
             if config.LOCAL_EXIT_MODEL or config.FINETUNED_EXIT_MODEL
             else "(not fine-tuned yet)")

    comparison.append({"candidate": history[-1]["text"][:50],
                       "expected": example["messages"][-1]["content"][:50],
                       "base": base[:50],
                       "tuned": tuned[:50]})
    print("candidate:", history[-1]["text"])
    print("expected :", example["messages"][-1]["content"])
    print("base     :", base)
    print("tuned    :", tuned)
    print("-" * 70)

pd.DataFrame(comparison)

The first two validation examples are refusals, the last two are
confirmations. What to look for: does the base model congratulate someone who
just withdrew, and does the fine-tuned one stop doing it **without** turning
every confirmation into a commiseration. Fixing one and breaking the other is
not an improvement.

## 10. Conclusions

Fill this in from the numbers above — these are the lines that get read.

| | |
|---|---|
| Baseline — always `continue` | 42.37% |
| Rules, no model | 88.14% |
| Main Agent, prompt v1 | _section 6_ |
| Main Agent, prompt v2 | _section 6_ |
| Main Agent, prompt v3 | _section 6_ |
| Ceiling — contradictory labels | 96.61% |
| Exit Advisor, base vs fine-tuned | _section 9_ |

Points worth making explicitly:

1. The test set is small (59 items). One item is **1.7%**, so a two-point
   difference between two prompts is noise, not an improvement.
2. Two pairs of identical candidate messages carry different labels, which caps
   accuracy at 96.61%. The remaining 3.39% is a property of the data, not of
   the model.
3. `end` is not one behaviour. Overall accuracy hides whether the 4 refusals are
   handled correctly — report that subset separately.
4. Seven regular expressions reach 88.14%. If a prompted LLM lands near that,
   it has not earned its cost on this task; what it buys is handling the
   phrasings the rules were never written for. Say which, with examples.
